# **MUD Task 3: Drug-Drug Interaction using Deep Learning**

**Authors:** Laia Jané and Elisa Müller



## Table of Contents

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Define paths
Define the paths to the data and utils in your Drive unit:

In [ ]:
# Elisa
utilsdir='/content/drive/MyDrive/LAB MUD/Task 3/07-DDI-nn'
evaluatordir='/content/drive/MyDrive/LAB MUD/Task 3/lab_resources/DDI/util'
trainfile='/content/drive/MyDrive/LAB MUD/Task 3/07-DDI-nn/train.pck'
validationfile='/content/drive/MyDrive/LAB MUD/Task 3/07-DDI-nn/devel.pck'
testfile='/content/drive/MyDrive/LAB MUD/Task 3/07-DDI-nn/test.pck'
validationdir='/content/drive/MyDrive/LAB MUD/Task 3/lab_resources/DDI/data/devel'
testdir='/content/drive/MyDrive/LAB MUD/Task 3/lab_resources/DDI/data/test'
modelname ='model.keras'
outfile ='out.txt'

In [ ]:
# # Laia
# utilsdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/07-DDI-nn'
# evaluatordir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/lab_resources/DDI/util'
# trainfile='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/07-DDI-nn/train.pck'
# validationfile='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/07-DDI-nn/devel.pck'
# testfile='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/07-DDI-nn/test.pck'
# validationdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/lab_resources/DDI/data/devel'
# testdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 3/lab_resources/DDI/data/test'
# modelname ='model.keras'
# outfile ='out.txt'

## 3. Set random seed

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

## 4. Add local paths and import modules

In [ ]:
!pip install -q transformers==4.33.2 accelerate
import sys
sys.path.insert(1,utilsdir) # Path to the utils folder on your Google Drive disk
sys.path.insert(1,evaluatordir) # Path to the evaluator folder on your Google Drive disk

ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons


In [ ]:
from contextlib import redirect_stdout

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification

from codemaps import *
from dataset import *

import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
FEATURES = ["input_ids", "attention_mask"]

## 5. Load train and validation data

In [ ]:
# load train and validation data
traindata = Dataset(trainfile)
valdata = Dataset(validationfile)

# create indexes from training data
max_len = 150
suf_len = 5
codes = Codemaps(traindata, max_len)

# encode datasets
Xt = codes.encode_words(traindata, features=FEATURES)
Yt = codes.encode_labels(traindata)
Xv = codes.encode_words(valdata, features=FEATURES)
Yv = codes.encode_labels(valdata)

n_tags = codes.get_n_labels()
max_len = codes.maxlen

## 4. Define `build_network`

In [ ]:
def build_model(n_labels):
    model = AutoModelForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=n_labels
    )
    return model

## 6. Create the model

In [ ]:
model = build_model(codes.get_n_labels())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print('Model loaded on', device)
print(model.config)

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ word_input (InputLayer)         │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_1  │ (None, 150, 128)       │       601,984 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 150, 128)       │       297,344 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 916,485 (3.50 MB)

 Trainable params: 916,485 (3.50 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Train the model

In [ ]:
train_labels = torch.tensor(np.argmax(Yt, axis=1), dtype=torch.long)
val_labels = torch.tensor(np.argmax(Yv, axis=1), dtype=torch.long)

train_dataset = TensorDataset(
    torch.tensor(Xt[0], dtype=torch.long),
    torch.tensor(Xt[1], dtype=torch.long),
    train_labels
)
val_dataset = TensorDataset(
    torch.tensor(Xv[0], dtype=torch.long),
    torch.tensor(Xv[1], dtype=torch.long),
    val_labels
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# 2-Stage Training Schedule
for epoch in range(3):
    # Stage 1 (Epoch 1): Freeze BERT, train only classifier
    if epoch == 0:
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}: Freezing BERT, training classifier only")
        print(f"{'='*60}")
        for param in model.bert.parameters():
            param.requires_grad = False
        # Only classifier head is trainable
        optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=2e-4)
    
    # Stage 2 (Epochs 2-3): Unfreeze BERT and fine-tune
    elif epoch == 1:
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}: Unfreezing BERT for fine-tuning")
        print(f"{'='*60}")
        for param in model.bert.parameters():
            param.requires_grad = True
        # Train all parameters with lower learning rate
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    
    # Training loop
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        input_ids_batch, attention_mask_batch, labels_batch = [t.to(device) for t in batch]
        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids_batch,
            attention_mask=attention_mask_batch,
            labels=labels_batch
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    print(f'Epoch {epoch+1} train loss: {train_loss:.4f}')

    # Validation loop
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids_batch, attention_mask_batch, labels_batch = [t.to(device) for t in batch]
            outputs = model(
                input_ids=input_ids_batch,
                attention_mask=attention_mask_batch,
                labels=labels_batch
            )
            val_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1)
            correct += (preds == labels_batch).sum().item()
            total += labels_batch.size(0)

    val_loss /= len(val_loader)
    print(f'Epoch {epoch+1} val loss: {val_loss:.4f}, val acc: {correct/total:.4f}')

print(f"\n{'='*60}")
print("Training complete!")
print(f"{'='*60}\n")

model.save_pretrained('bert_model')
codes.save('bert_model')

Epoch 1/15
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: word_input
Received: inputs=('Tensor(shape=(None, 150))',)
  warnings.warn(msg)
724/724 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step - accuracy: 0.8508 - loss: 0.5192 - val_accuracy: 0.8427 - val_loss: 0.4677 - learning_rate: 0.0010
Epoch 2/15
724/724 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.8561 - loss: 0.3806 - val_accuracy: 0.8338 - val_loss: 0.4626 - learning_rate: 0.0010
Epoch 3/15
724/724 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.8642 - loss: 0.3487 - val_accuracy: 0.8497 - val_loss: 0.4722 - learning_rate: 0.0010
Epoch 4/15
724/724 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8705 - loss: 0.3316
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
724/724 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.8710 - loss: 0.3292 - val_accuracy: 0.8490 - val_loss: 0.4880 - learning_r

# 8. Predict

In [ ]:
#import sys
import evaluator

In [ ]:
def output_interactions(data, preds, outfile) :

   #print(testdata[0])
   outf = open(outfile, 'w')
   for exmp,tag in zip(data.sentences(),preds) :
      sid = exmp['sid']
      e1 = exmp['e1']
      e2 = exmp['e2']
      if tag!='null' :
         print(sid, e1, e2, tag, sep="|", file=outf)

   outf.close()

## 9. Evaluation function

In [ ]:
## --------- Evaluator -----------
def evaluation(datadir,outfile) :
   evaluator.evaluate("DDI", datadir, outfile)


In [ ]:
# Validation data
X = codes.encode_words(valdata, features=FEATURES)
model.eval()
with torch.no_grad():
    inputs = {
        'input_ids': torch.tensor(X[0], dtype=torch.long).to(device),
        'attention_mask': torch.tensor(X[1], dtype=torch.long).to(device)
    }
    outputs = model(**inputs)
    preds = outputs.logits.argmax(dim=-1).cpu().numpy()

Y = [codes.idx2label(int(i)) for i in preds]

# extract entities
output_interactions(valdata, Y, outfile)

# evaluate
evaluation(validationdir,outfile)

  1/145 ━━━━━━━━━━━━━━━━━━━━ 26s 185ms/step

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: word_input
Received: inputs=('Tensor(shape=(32, 150))',)
  warnings.warn(msg)


145/145 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
advise             55	  36	  86	  91	 141	60.4%	39.0%	47.4%
effect            106	 157	 206	 263	 312	40.3%	34.0%	36.9%
int                 1	   3	  27	   4	  28	25.0%	3.6%	6.2%
mechanism          16	  15	 245	  31	 261	51.6%	6.1%	11.0%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	44.3%	20.7%	25.4%
------------------------------------------------------------------------------
m.avg             178	 211	 564	 389	 742	45.8%	24.0%	31.5%
m.avg(no class)   186	 203	 556	 389	 742	47.8%	25.1%	32.9%


In [ ]:
# Test data
testdata = Dataset(testfile)

X = codes.encode_words(testdata, features=FEATURES)
model.eval()
with torch.no_grad():
    inputs = {
        'input_ids': torch.tensor(X[0], dtype=torch.long).to(device),
        'attention_mask': torch.tensor(X[1], dtype=torch.long).to(device)
    }
    outputs = model(**inputs)
    preds = outputs.logits.argmax(dim=-1).cpu().numpy()

Y = [codes.idx2label(int(i)) for i in preds]

# extract entities
output_interactions(testdata, Y, outfile)

# evaluate
evaluation(testdir,outfile)

180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
advise             45	  35	 164	  80	 209	56.2%	21.5%	31.1%
effect            100	 122	 186	 222	 286	45.0%	35.0%	39.4%
int                 1	   3	  24	   4	  25	25.0%	4.0%	6.9%
mechanism          30	  20	 310	  50	 340	60.0%	8.8%	15.4%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	46.6%	17.3%	23.2%
------------------------------------------------------------------------------
m.avg             176	 180	 684	 356	 860	49.4%	20.5%	28.9%
m.avg(no class)   192	 164	 668	 356	 860	53.9%	22.3%	31.6%
